# Analyse des campagnes de simulation et d'entraînement

Ce script lit l'ensemble des fichiers Parquet segmentés, des résumés CSV, des manifestes de recherche combinatoire et des historiques d'entraînement produits dans `data/` et `weights/`, puis produit un large éventail de graphiques couvrant les métriques macro, micro,
comportementales et de complexité du projet.

In [ ]:
import glob
import json
import os
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from bokeh.io import output_file, save
from bokeh.palettes import Viridis256
from bokeh.plotting import figure as bokeh_figure
from bokeh.transform import linear_cmap
import re

sns.set_theme(style="whitegrid")

DATA_DIR = "data"
WEIGHTS_DIR = "weights"
FIGURE_DIR = "figures"
os.makedirs(FIGURE_DIR, exist_ok=True)

_POINTS_TABLE = {
    "3": 3, "4": 4, "5": 5, "6": 6, "7": 7, "8": 8, "9": 9, "10": 10,
    "J": 11, "Q": 12, "K": 13, "A": 14, "2": 15, "JOKER": 16,
}


def gini(values: List[float]) -> float:
    ordered = sorted(values)
    n = len(ordered)
    if n == 0:
        return 0.0
    total = sum(ordered)
    if total == 0:
        return 0.0
    cumulative = sum((i + 1) * v for i, v in enumerate(ordered))
    return (2 * cumulative) / (n * total) - (n + 1) / n


In [ ]:
def load_segmented_parquet(pattern: str = "*.parquet") -> pd.DataFrame:
    paths = sorted(glob.glob(os.path.join(DATA_DIR, pattern)))
    frames = []
    for path in paths:
        frame = pd.read_parquet(path)
        frame["source_file"] = os.path.basename(path)
        frames.append(frame)
    if not frames:
        return pd.DataFrame(
            columns=["event_type", "timestamp", "game_id", "round_id", "state_hash", "payload", "source_file"]
        )
    return pd.concat(frames, ignore_index=True)


def expand_payload(df: pd.DataFrame, event_type: str) -> pd.DataFrame:
    subset = df[df["event_type"] == event_type].copy()
    if subset.empty:
        return subset
    payloads = subset["payload"].apply(json.loads)
    payload_df = pd.json_normalize(payloads)
    payload_df.index = subset.index
    return pd.concat([subset.drop(columns=["payload"]), payload_df], axis=1)


def load_summary_csvs(pattern: str = "*summary.csv") -> pd.DataFrame:
    paths = sorted(glob.glob(os.path.join(DATA_DIR, pattern)))
    frames = [pd.read_csv(path) for path in paths]
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def load_grid_manifests(pattern: str = "*manifest*.csv") -> pd.DataFrame:
    paths = sorted(glob.glob(os.path.join(DATA_DIR, pattern)))
    frames = [pd.read_csv(path) for path in paths]
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def load_training_histories(pattern: str = "*history.csv") -> pd.DataFrame:
    paths = sorted(glob.glob(os.path.join(WEIGHTS_DIR, pattern)))
    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        frame["model_file"] = os.path.basename(path)
        frames.append(frame)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


In [ ]:
events_df = load_segmented_parquet()
finished_df = expand_payload(events_df, "EventPlayerFinished")
round_start_df = expand_payload(events_df, "EventRoundStart")
action_played_df = expand_payload(events_df, "EventActionPlayed")
action_request_df = expand_payload(events_df, "EventActionRequest")
trick_start_df = expand_payload(events_df, "EventTrickStart")
round_end_df = expand_payload(events_df, "EventRoundEnd")
rule_triggered_df = expand_payload(events_df, "EventRuleTriggered")

summary_df = load_summary_csvs()
manifest_df = load_grid_manifests()
history_df = load_training_histories()


## Distribution des points de victoire par rang de sortie (violin plot)

In [ ]:
if not finished_df.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.violinplot(data=finished_df, x="rank", y="vp_earned", ax=ax, inner="quartile")
    ax.set_xlabel("Rang de sortie")
    ax.set_ylabel("Points de victoire")
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURE_DIR, "vp_by_rank_violin.png"), dpi=150)
    plt.show()


## Taux de victoire par identifiant de joueur, avec intervalle de confiance

In [ ]:
if not finished_df.empty:
    finished_df["is_winner"] = finished_df["rank"] == 0
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=finished_df, x="player_id", y="is_winner", errorbar=("ci", 95), ax=ax)
    ax.set_xlabel("Identifiant de joueur")
    ax.set_ylabel("Taux de victoire (rang 0)")
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURE_DIR, "win_rate_by_player_ci.png"), dpi=150)
    plt.show()


## Taux de passe sous-optimal par identifiant de joueur, avec intervalle de confiance

In [ ]:
if not action_played_df.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=action_played_df, x="player_id", y="was_suboptimal", errorbar=("ci", 95), ax=ax)
    ax.set_xlabel("Identifiant de joueur")
    ax.set_ylabel("Taux de passe sous-optimal")
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURE_DIR, "suboptimal_pass_rate_ci.png"), dpi=150)
    plt.show()


## Matrice de transition des rôles d'une manche à la suivante (heatmap)

In [ ]:
role_transition_matrix = pd.DataFrame()
if not round_end_df.empty and "roles_by_player" in round_end_df.columns:
    transition_counts: Dict[str, Dict[str, int]] = {}
    grouped = round_end_df.sort_values(["game_id", "round_id"]).groupby("game_id")
    for _, group in grouped:
        roles_sequence = group["roles_by_player"].tolist()
        for earlier, later in zip(roles_sequence, roles_sequence[1:]):
            for pid, role_a in earlier.items():
                role_b = later.get(pid)
                if role_b is None:
                    continue
                transition_counts.setdefault(role_a, {}).setdefault(role_b, 0)
                transition_counts[role_a][role_b] += 1

    roles_order = ["ROLE_PRESIDENT", "ROLE_VICE_PRESIDENT", "ROLE_NEUTRAL", "ROLE_VICE_SCUM", "ROLE_SCUM"]
    role_transition_matrix = pd.DataFrame(0.0, index=roles_order, columns=roles_order)
    for role_a, counts in transition_counts.items():
        total = sum(counts.values())
        for role_b, count in counts.items():
            if role_a in role_transition_matrix.index and role_b in role_transition_matrix.columns:
                role_transition_matrix.loc[role_a, role_b] = count / total if total else 0.0

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(role_transition_matrix, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=ax)
    ax.set_xlabel("Rôle manche suivante")
    ax.set_ylabel("Rôle manche courante")
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURE_DIR, "role_transition_heatmap.png"), dpi=150)
    plt.show()


## Diagramme d'accords des transitions de rôles

In [ ]:
def draw_chord_diagram(matrix: pd.DataFrame, path: str) -> None:
    labels = list(matrix.index)
    n = len(labels)
    if n == 0:
        return
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
    fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"aspect": "equal"})
    radius = 1.0
    positions = {label: (radius * np.cos(angle), radius * np.sin(angle)) for label, angle in zip(labels, angles)}
    for label, (x, y) in positions.items():
        ax.scatter([x], [y], s=200, zorder=3)
        ax.text(x * 1.15, y * 1.15, label, ha="center", va="center", fontsize=8)
    max_value = matrix.values.max() if matrix.values.size else 1.0
    for row_label in labels:
        for col_label in labels:
            value = matrix.loc[row_label, col_label]
            if value <= 0 or row_label == col_label:
                continue
            x0, y0 = positions[row_label]
            x1, y1 = positions[col_label]
            curve = np.array([[x0, y0], [0.0, 0.0], [x1, y1]])
            t = np.linspace(0, 1, 50)
            bezier = (
                ((1 - t)[:, None] ** 2) * curve[0]
                + 2 * (1 - t)[:, None] * t[:, None] * curve[1]
                + (t ** 2)[:, None] * curve[2]
            )
            ax.plot(bezier[:, 0], bezier[:, 1], alpha=min(1.0, value / max_value), linewidth=1.5)
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.axis("off")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.show()


if not role_transition_matrix.empty:
    draw_chord_diagram(role_transition_matrix, os.path.join(FIGURE_DIR, "role_transition_chord.png"))


## Densité miroir des points de victoire, PRESIDENT contre SCUM (pyramide des âges)

In [ ]:
if not finished_df.empty and not round_end_df.empty and "roles_by_player" in round_end_df.columns:
    role_lookup: Dict[tuple, str] = {}
    for _, row in round_end_df.iterrows():
        for pid, role in row["roles_by_player"].items():
            role_lookup[(row["game_id"], row["round_id"], int(pid))] = role
    finished_df["role"] = finished_df.apply(
        lambda r: role_lookup.get((r["game_id"], r["round_id"], r["player_id"])), axis=1
    )
    president_vp = finished_df[finished_df["role"] == "ROLE_PRESIDENT"]["vp_earned"]
    scum_vp = finished_df[finished_df["role"] == "ROLE_SCUM"]["vp_earned"]
    if not president_vp.empty or not scum_vp.empty:
        lower = min(president_vp.min() if not president_vp.empty else 0, scum_vp.min() if not scum_vp.empty else 0)
        upper = max(president_vp.max() if not president_vp.empty else 1, scum_vp.max() if not scum_vp.empty else 1)
        bins = np.linspace(lower, upper, 20)
        president_counts, _ = np.histogram(president_vp, bins=bins)
        scum_counts, _ = np.histogram(scum_vp, bins=bins)
        bin_centers = (bins[:-1] + bins[1:]) / 2
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.barh(bin_centers, president_counts, height=(bins[1] - bins[0]), label="ROLE_PRESIDENT")
        ax.barh(bin_centers, -scum_counts, height=(bins[1] - bins[0]), label="ROLE_SCUM")
        ax.set_xlabel("Nombre de sorties")
        ax.set_ylabel("Points de victoire")
        ax.legend()
        fig.tight_layout()
        fig.savefig(os.path.join(FIGURE_DIR, "vp_mirror_density.png"), dpi=150)
        plt.show()


## Histogramme de l'indice de Gini de la puissance de main initiale

In [ ]:
if not round_start_df.empty and "initial_hands" in round_start_df.columns:
    gini_records = []
    for _, row in round_start_df.iterrows():
        hands = row["initial_hands"]
        for pid, cards in hands.items():
            power_sum = sum(_POINTS_TABLE.get(card["rank"], 0) for card in cards if card["rank"] != "JOKER")
            gini_records.append({"game_id": row["game_id"], "round_id": row["round_id"], "player_id": pid, "hand_power": power_sum})
    hand_power_df = pd.DataFrame(gini_records)
    if not hand_power_df.empty:
        gini_per_round = (
            hand_power_df.groupby(["game_id", "round_id"])["hand_power"]
            .apply(lambda values: gini(values.tolist()))
            .reset_index(name="gini")
        )
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.histplot(gini_per_round["gini"], kde=True, ax=ax)
        ax.set_xlabel("Indice de Gini de la puissance de main initiale")
        ax.set_ylabel("Nombre de manches")
        fig.tight_layout()
        fig.savefig(os.path.join(FIGURE_DIR, "gini_hand_power_histogram.png"), dpi=150)
        plt.show()


## Facteur de branchement moyen en fonction du nombre de joueurs, par profil d'agent (bandes d'incertitude)

In [ ]:
if not manifest_df.empty and "branching_factor_average" in manifest_df.columns:
    grouped = (
        manifest_df.groupby(["agent_profile", "player_count"])["branching_factor_average"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    grouped["sem"] = grouped["std"].fillna(0) / grouped["count"].clip(lower=1) ** 0.5
    fig, ax = plt.subplots(figsize=(8, 5))
    for profile, group in grouped.groupby("agent_profile"):
        group = group.sort_values("player_count")
        ax.plot(group["player_count"], group["mean"], marker="o", label=profile)
        ax.fill_between(group["player_count"], group["mean"] - group["sem"], group["mean"] + group["sem"], alpha=0.2)
    ax.set_xlabel("Nombre de joueurs")
    ax.set_ylabel("Facteur de branchement moyen")
    ax.legend(title="Profil d'agent")
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURE_DIR, "branching_factor_vs_player_count.png"), dpi=150)
    plt.show()


## Distribution du nombre d'options légales par profil d'agent (ridgeline)

In [ ]:
if not action_request_df.empty:
    profiles = action_request_df["source_file"].unique()
    fig, axes = plt.subplots(len(profiles), 1, figsize=(8, 1.2 * max(len(profiles), 1)), sharex=True)
    axes = np.atleast_1d(axes)
    for ax, profile in zip(axes, profiles):
        subset = action_request_df.loc[action_request_df["source_file"] == profile, "legal_action_count"]
        sns.kdeplot(subset, fill=True, ax=ax, alpha=0.6)
        ax.set_ylabel("")
        ax.text(0.01, 0.5, profile, transform=ax.transAxes, va="center", fontsize=8)
    axes[-1].set_xlabel("Nombre d'options légales")
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURE_DIR, "legal_action_count_ridgeline.png"), dpi=150)
    plt.show()


## Occurrences de combinaisons jouées par taille et puissance résultante (bubble plot)

In [ ]:
if not action_played_df.empty and "resulting_power" in action_played_df.columns:
    played = action_played_df[action_played_df["action_type"] == "ACTION_PLAY"].copy()
    played["combo_size"] = played["cards_played"].apply(len)
    bubble_data = played.groupby(["combo_size", "resulting_power"]).size().reset_index(name="count")
    if not bubble_data.empty:
        fig = px.scatter(
            bubble_data, x="combo_size", y="resulting_power", size="count", color="count",
            labels={
                "combo_size": "Taille de la combinaison",
                "resulting_power": "Puissance résultante",
                "count": "Occurrences",
            },
        )
        fig.write_html(os.path.join(FIGURE_DIR, "combo_power_bubble.html"))
        fig.show()


## Comparaison normalisée des profils d'agents sur plusieurs métriques (radar)

In [ ]:
if not manifest_df.empty:
    metric_columns = [
        c for c in [
            "branching_factor_average", "action_space_entropy", "e_rev_volatility",
            "trick_length_average", "gini_initial_hand_power_mean",
        ]
        if c in manifest_df.columns
    ]
    if metric_columns:
        radar_data = manifest_df.groupby("agent_profile")[metric_columns].mean()
        ranges = (radar_data.max() - radar_data.min()).replace(0, 1)
        normalized = (radar_data - radar_data.min()) / ranges
        categories = metric_columns + [metric_columns[0]]
        fig = go.Figure()
        for profile in normalized.index:
            values = normalized.loc[profile].tolist()
            values.append(values[0])
            fig.add_trace(go.Scatterpolar(r=values, theta=categories, fill="toself", name=profile))
        fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 1])))
        fig.write_html(os.path.join(FIGURE_DIR, "agent_profile_radar.html"))
        fig.show()


## Distribution des tailles de combinaison jouées, par fichier source

In [ ]:
if not action_played_df.empty:
    played = action_played_df[action_played_df["action_type"] == "ACTION_PLAY"].copy()
    played["combo_size"] = played["cards_played"].apply(len)
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.countplot(data=played, x="combo_size", hue="source_file", ax=ax)
    ax.set_xlabel("Taille de combinaison")
    ax.set_ylabel("Nombre de poses")
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURE_DIR, "combo_size_distribution.png"), dpi=150)
    plt.show()


## Nombre d'actions par manche, par fichier source (violin plot)

In [ ]:
if not action_played_df.empty:
    actions_per_round = (
        action_played_df.groupby(["source_file", "game_id", "round_id"]).size().reset_index(name="actions")
    )
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.violinplot(data=actions_per_round, x="source_file", y="actions", ax=ax, inner="quartile")
    ax.set_xlabel("Fichier source")
    ax.set_ylabel("Actions par manche")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURE_DIR, "actions_per_round_violin.png"), dpi=150)
    plt.show()


## Volatilité de la Révolution en fonction du nombre de joueurs (régression)

In [ ]:
if not manifest_df.empty and "e_rev_volatility" in manifest_df.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.regplot(data=manifest_df, x="player_count", y="e_rev_volatility", ax=ax, scatter_kws={"alpha": 0.4})
    ax.set_xlabel("Nombre de joueurs")
    ax.set_ylabel("Bascules de révolution par manche")
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURE_DIR, "e_rev_volatility_vs_players.png"), dpi=150)
    plt.show()


## Corrélation entre la position d'ouverture et le rang de sortie (régression)

In [ ]:
if not trick_start_df.empty and not finished_df.empty:
    openers = (
        trick_start_df[trick_start_df["trick_index"] == 0][["game_id", "round_id", "opener_id"]]
        .drop_duplicates(["game_id", "round_id"])
    )
    merged = finished_df.merge(openers, on=["game_id", "round_id"], how="inner")
    if not merged.empty:
        correlation = merged["opener_id"].corr(merged["rank"])
        fig, ax = plt.subplots(figsize=(7, 6))
        sns.regplot(data=merged, x="opener_id", y="rank", ax=ax, scatter_kws={"alpha": 0.3})
        ax.set_xlabel("Identifiant du joueur ouvrant le premier pli")
        ax.set_ylabel("Rang de sortie")
        ax.set_title(f"r = {correlation:.3f}")
        fig.tight_layout()
        fig.savefig(os.path.join(FIGURE_DIR, "opening_position_rank_regression.png"), dpi=150)
        plt.show()


## Heatmap du facteur de branchement moyen selon le nombre de joueurs et le préset de règles

In [ ]:
if not manifest_df.empty and "rule_preset" in manifest_df.columns:
    pivot = manifest_df.pivot_table(
        index="player_count", columns="rule_preset", values="branching_factor_average", aggfunc="mean"
    )
    if not pivot.empty:
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(pivot, annot=True, fmt=".2f", cmap="magma", ax=ax)
        ax.set_xlabel("Configuration de règles")
        ax.set_ylabel("Nombre de joueurs")
        fig.tight_layout()
        fig.savefig(os.path.join(FIGURE_DIR, "branching_factor_heatmap.png"), dpi=150)
        plt.show()


## Contour de densité : Gini de la main initiale contre facteur de branchement

In [ ]:
if not manifest_df.empty and {"gini_initial_hand_power_mean", "branching_factor_average"}.issubset(manifest_df.columns):
    fig = px.density_contour(
        manifest_df, x="gini_initial_hand_power_mean", y="branching_factor_average", color="agent_profile",
        labels={
            "gini_initial_hand_power_mean": "Indice de Gini moyen de la main initiale",
            "branching_factor_average": "Facteur de branchement moyen",
        },
    )
    fig.write_html(os.path.join(FIGURE_DIR, "gini_branching_contour.html"))
    fig.show()


## Nuage de points interactif : manches par partie contre facteur de branchement (Bokeh)

In [ ]:
if not manifest_df.empty and "rounds_per_game" in manifest_df.columns:
    output_file(os.path.join(FIGURE_DIR, "rounds_vs_branching_bokeh.html"))
    color_mapper = linear_cmap(
        "player_count", Viridis256, manifest_df["player_count"].min(), manifest_df["player_count"].max()
    )
    bokeh_fig = bokeh_figure(
        x_axis_label="Manches par partie", y_axis_label="Facteur de branchement moyen", width=800, height=500,
    )
    bokeh_fig.scatter(
        "rounds_per_game", "branching_factor_average", source=manifest_df, size=8, color=color_mapper, alpha=0.6,
    )
    save(bokeh_fig)


## Fréquence des déclenchements de règles avancées

In [ ]:
if not rule_triggered_df.empty:
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.countplot(
        data=rule_triggered_df, y="rule_name",
        order=rule_triggered_df["rule_name"].value_counts().index, ax=ax,
    )
    ax.set_xlabel("Occurrences")
    ax.set_ylabel("Règle déclenchée")
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURE_DIR, "rule_trigger_counts.png"), dpi=150)
    plt.show()


## Courbes d'apprentissage des modèles entraînés, avec bandes d'incertitude

In [ ]:
if not history_df.empty:
    fig, ax = plt.subplots(figsize=(9, 5))
    for model_file, group in history_df.groupby("model_file"):
        group = group.sort_values("round_index")
        window = max(1, len(group) // 100)
        rolling_mean = group["vp"].rolling(window=window, min_periods=1).mean()
        rolling_std = group["vp"].rolling(window=window, min_periods=1).std().fillna(0)
        ax.plot(group["round_index"], rolling_mean, label=model_file)
        ax.fill_between(
            group["round_index"], rolling_mean - rolling_std, rolling_mean + rolling_std, alpha=0.2,
        )
    ax.set_xlabel("Index de manche d'entraînement")
    ax.set_ylabel("Point de victoire (moyenne glissante)")
    ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURE_DIR, "training_learning_curves.png"), dpi=150)
    plt.show()


## Chargement des événements transactionnels additionnels (Putsch, échange, interception) et des résultats d'évaluation comparative

In [ ]:
ask_putsch_df = expand_payload(events_df, "EventAskPutsch")
putsch_invoked_df = expand_payload(events_df, "EventPutschInvoked")
exchange_df = expand_payload(events_df, "EventExchange")
interception_broadcast_df = expand_payload(events_df, "EventInterceptionBroadcast")
interception_resolved_df = expand_payload(events_df, "EventInterceptionResolved")

In [ ]:
def load_evaluation_csvs(pattern: str = "*.csv") -> pd.DataFrame:
    paths = sorted(glob.glob(os.path.join(DATA_DIR, pattern)))
    paths = [p for p in paths if "summary" not in os.path.basename(p) and "manifest" not in os.path.basename(p)]
    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        frame["source_file"] = os.path.basename(path)
        frames.append(frame)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def parse_learning_rate(model_file: str) -> float:
    match = re.search(r"learnRate([0-9pm]+)_", model_file)
    if not match:
        return float("nan")
    token = match.group(1).replace("p", ".").replace("m", "-")
    try:
        return float(token)
    except ValueError:
        return float("nan")


evaluation_df = load_evaluation_csvs()
if not history_df.empty:
    history_df["learning_rate"] = history_df["model_file"].apply(parse_learning_rate)


## Efficacité du Putsch : taux de victoire du rôle SCUM sollicité, selon invocation ou non, avec intervalle de confiance

In [ ]:
if not ask_putsch_df.empty and not finished_df.empty and not putsch_invoked_df.empty:
    invoked_rounds = set(zip(putsch_invoked_df["game_id"], putsch_invoked_df["round_id"]))
    scum_by_round = {
        (row["game_id"], row["round_id"]): row["player_id"] for _, row in ask_putsch_df.iterrows()
    }
    records = []
    for _, row in finished_df.iterrows():
        key = (row["game_id"], row["round_id"])
        scum_id = scum_by_round.get(key)
        if scum_id is None or row["player_id"] != scum_id:
            continue
        category = "Putsch invoqué" if key in invoked_rounds else "Putsch non invoqué"
        records.append({"category": category, "is_winner": row["rank"] == 0})
    putsch_df = pd.DataFrame(records)
    if not putsch_df.empty:
        fig, ax = plt.subplots(figsize=(7, 5))
        sns.barplot(data=putsch_df, x="category", y="is_winner", errorbar=("ci", 95), ax=ax)
        ax.set_xlabel("Décision de Putsch")
        ax.set_ylabel("Taux de victoire du rôle SCUM sollicité")
        fig.tight_layout()
        fig.savefig(os.path.join(FIGURE_DIR, "putsch_efficiency_ci.png"), dpi=150)
        plt.show()


## Poids de la taxe d'échange contre point de victoire du destinataire, par fichier source (régression)

In [ ]:
if not exchange_df.empty and not finished_df.empty and "cards" in exchange_df.columns:
    vp_lookup = {
        (row["game_id"], row["round_id"], row["player_id"]): row["vp_earned"]
        for _, row in finished_df.iterrows()
    }
    records = []
    for _, row in exchange_df.iterrows():
        key = (row["game_id"], row["round_id"], row["to_player"])
        vp = vp_lookup.get(key)
        if vp is None:
            continue
        weight = sum(_POINTS_TABLE.get(card["rank"], 0) for card in row["cards"])
        records.append({"tax_weight": weight, "vp_earned": vp, "source_file": row["source_file"]})
    tax_df = pd.DataFrame(records)
    if not tax_df.empty:
        fig = px.scatter(
            tax_df, x="tax_weight", y="vp_earned", color="source_file", trendline="ols",
            labels={"tax_weight": "Poids en points des cartes échangées", "vp_earned": "Point de victoire du destinataire"},
        )
        fig.write_html(os.path.join(FIGURE_DIR, "tax_weight_vp_regression.html"))
        fig.show()


## Taux d'interception manquée par fichier source, avec intervalle de confiance

In [ ]:
if not interception_broadcast_df.empty and not interception_resolved_df.empty:
    broadcasts = interception_broadcast_df.sort_values(["source_file", "game_id", "round_id", "timestamp"])
    resolutions = interception_resolved_df.sort_values(["source_file", "game_id", "round_id", "timestamp"])
    merged = pd.merge_asof(
        broadcasts, resolutions[["source_file", "game_id", "round_id", "timestamp", "interceptor_id"]],
        on="timestamp", by=["source_file", "game_id", "round_id"], direction="forward",
    )
    merged["missed"] = merged["interceptor_id"].isna()
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=merged, x="source_file", y="missed", errorbar=("ci", 95), ax=ax)
    ax.set_xlabel("Fichier source")
    ax.set_ylabel("Taux d'interception manquée (approché)")
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURE_DIR, "missed_interception_rate_ci.png"), dpi=150)
    plt.show()


## Distribution du nombre de joueurs sautés par déclenchement du Saut de Tour (histogramme)

In [ ]:
if not rule_triggered_df.empty and "magnitude" in rule_triggered_df.columns:
    skip_events = rule_triggered_df[rule_triggered_df["rule_name"] == "SKIP_TURN"].dropna(subset=["magnitude"])
    if not skip_events.empty:
        fig, ax = plt.subplots(figsize=(7, 5))
        sns.histplot(skip_events["magnitude"], discrete=True, ax=ax)
        ax.set_xlabel("Nombre de joueurs sautés")
        ax.set_ylabel("Occurrences")
        fig.tight_layout()
        fig.savefig(os.path.join(FIGURE_DIR, "skip_turn_magnitude_histogram.png"), dpi=150)
        plt.show()


## Ratio de capture de points par joueur, en fonction des points engagés (nuage de points)

In [ ]:
if not action_played_df.empty and not trick_start_df.empty:
    plays = action_played_df[action_played_df["action_type"] == "ACTION_PLAY"].copy()
    plays["points_spent"] = plays["cards_played"].apply(
        lambda cards: sum(_POINTS_TABLE.get(c["rank"], 0) for c in cards)
    )
    spent_by_player = plays.groupby(["source_file", "player_id"])["points_spent"].sum().reset_index()
    if not spent_by_player.empty:
        fig = px.scatter(
            spent_by_player, x="player_id", y="points_spent", color="source_file", size="points_spent",
            labels={"player_id": "Identifiant de joueur", "points_spent": "Points engagés (cumulés)"},
        )
        fig.write_html(os.path.join(FIGURE_DIR, "points_spent_by_player.html"))
        fig.show()


## Distribution du rang de la première carte de rang 2 ou supérieur jouée par joueur (boîtes à moustaches)

In [ ]:
if not action_played_df.empty:
    plays = action_played_df[action_played_df["action_type"] == "ACTION_PLAY"].copy()
    plays = plays.sort_values(["source_file", "game_id", "round_id", "timestamp"])
    plays["play_index"] = plays.groupby(["source_file", "game_id", "round_id", "player_id"]).cumcount() + 1
    plays["has_high_card"] = plays["cards_played"].apply(
        lambda cards: any(c["rank"] == "2" for c in cards)
    )
    first_high = (
        plays[plays["has_high_card"]]
        .groupby(["source_file", "game_id", "round_id", "player_id"])["play_index"]
        .min()
        .reset_index(name="card_ttl")
    )
    if not first_high.empty:
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.boxplot(data=first_high, x="player_id", y="card_ttl", ax=ax)
        ax.set_xlabel("Identifiant de joueur")
        ax.set_ylabel("Nombre de poses avant la première carte de rang 2")
        fig.tight_layout()
        fig.savefig(os.path.join(FIGURE_DIR, "card_ttl_boxplot.png"), dpi=150)
        plt.show()


## Indice d'agressivité à l'ouverture : puissance posée contre puissance moyenne de la main (régression)

In [ ]:
if not trick_start_df.empty and not action_played_df.empty and not round_start_df.empty:
    openers = trick_start_df[trick_start_df["trick_index"] == 0][
        ["source_file", "game_id", "round_id", "opener_id"]
    ].drop_duplicates(["source_file", "game_id", "round_id"])
    hand_power_records = []
    for _, row in round_start_df.iterrows():
        for pid, cards in row["initial_hands"].items():
            power_sum = sum(_POINTS_TABLE.get(card["rank"], 0) for card in cards if card["rank"] != "JOKER")
            hand_power_records.append({
                "source_file": row["source_file"], "game_id": row["game_id"], "round_id": row["round_id"],
                "player_id": int(pid), "hand_power": power_sum,
            })
    hand_power_df = pd.DataFrame(hand_power_records)
    opening_plays = action_played_df[
        (action_played_df["action_type"] == "ACTION_PLAY") & (action_played_df["resulting_power"].notna())
    ]
    merged = opening_plays.merge(openers, on=["source_file", "game_id", "round_id"])
    merged = merged[merged["player_id"] == merged["opener_id"]]
    merged = merged.merge(hand_power_df, on=["source_file", "game_id", "round_id", "player_id"])
    merged = merged[merged["hand_power"] > 0]
    if not merged.empty:
        fig = px.scatter(
            merged, x="hand_power", y="resulting_power", color="source_file", trendline="ols",
            labels={"hand_power": "Puissance totale de la main à l'ouverture", "resulting_power": "Puissance de la carte posée"},
        )
        fig.write_html(os.path.join(FIGURE_DIR, "opening_aggressiveness_regression.html"))
        fig.show()


## Comparaison radar des profils évalués sur plusieurs métriques normalisées (évaluation comparative)

In [ ]:
if not evaluation_df.empty:
    eval_metric_columns = [c for c in ["cumulative_vp", "president_rounds"] if c in evaluation_df.columns]
    if eval_metric_columns:
        radar_eval = evaluation_df.groupby("profile")[eval_metric_columns].mean()
        eval_ranges = (radar_eval.max() - radar_eval.min()).replace(0, 1)
        normalized_eval = (radar_eval - radar_eval.min()) / eval_ranges
        categories_eval = eval_metric_columns + [eval_metric_columns[0]]
        fig = go.Figure()
        for profile in normalized_eval.index:
            values = normalized_eval.loc[profile].tolist()
            values.append(values[0])
            fig.add_trace(go.Scatterpolar(r=values, theta=categories_eval, fill="toself", name=profile))
        fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 1])))
        fig.write_html(os.path.join(FIGURE_DIR, "evaluation_profile_radar.html"))
        fig.show()


## Distribution du point de victoire cumulé par profil évalué (violin plot)

In [ ]:
if not evaluation_df.empty and "cumulative_vp" in evaluation_df.columns:
    fig, ax = plt.subplots(figsize=(9, 5))
    sns.violinplot(data=evaluation_df, x="profile", y="cumulative_vp", ax=ax, inner="quartile")
    ax.set_xlabel("Profil évalué")
    ax.set_ylabel("Point de victoire cumulé par partie")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURE_DIR, "evaluation_vp_violin.png"), dpi=150)
    plt.show()


## Taux de manches présidentielles par profil évalué, avec intervalle de confiance

In [ ]:
if not evaluation_df.empty and "president_rounds" in evaluation_df.columns and "rounds_per_game" in evaluation_df.columns:
    evaluation_df["president_rate"] = evaluation_df["president_rounds"] / evaluation_df["rounds_per_game"].clip(lower=1)
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=evaluation_df, x="profile", y="president_rate", errorbar=("ci", 95), ax=ax)
    ax.set_xlabel("Profil évalué")
    ax.set_ylabel("Taux de manches terminées au rôle PRESIDENT")
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGURE_DIR, "evaluation_president_rate_ci.png"), dpi=150)
    plt.show()


## Performance finale d'entraînement en fonction du taux d'apprentissage (boîtes à moustaches et contour de densité)

In [ ]:
if not history_df.empty and "learning_rate" in history_df.columns:
    final_performance = []
    for model_file, group in history_df.groupby("model_file"):
        group = group.sort_values("round_index")
        tail = group.tail(max(1, len(group) // 20))
        final_performance.append({
            "model_file": model_file,
            "learning_rate": group["learning_rate"].iloc[0],
            "final_vp_mean": tail["vp"].mean(),
            "total_rounds": group["round_index"].max(),
        })
    final_performance_df = pd.DataFrame(final_performance).dropna(subset=["learning_rate"])
    if not final_performance_df.empty:
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.boxplot(data=final_performance_df, x="learning_rate", y="final_vp_mean", ax=ax)
        ax.set_xlabel("Taux d'apprentissage")
        ax.set_ylabel("VP moyen final (derniers 5% de manches)")
        fig.tight_layout()
        fig.savefig(os.path.join(FIGURE_DIR, "learning_rate_final_performance_boxplot.png"), dpi=150)
        plt.show()

        fig2 = px.density_contour(
            final_performance_df, x="learning_rate", y="total_rounds", z="final_vp_mean", histfunc="avg",
            labels={"learning_rate": "Taux d'apprentissage", "total_rounds": "Manches entraînées", "final_vp_mean": "VP moyen final"},
        )
        fig2.write_html(os.path.join(FIGURE_DIR, "learning_rate_rounds_contour.html"))
        fig2.show()


## Nombre de joueurs contre manches par partie, taille et couleur selon le facteur de branchement moyen (Nuage de bulles)

In [ ]:
if not manifest_df.empty and {"player_count", "rounds_per_game", "branching_factor_average"}.issubset(manifest_df.columns):
    bubble_manifest = manifest_df.groupby(["player_count", "rounds_per_game"])["branching_factor_average"].mean().reset_index()
    if not bubble_manifest.empty:
        fig = px.scatter(
            bubble_manifest, x="player_count", y="rounds_per_game", size="branching_factor_average",
            color="branching_factor_average",
            labels={
                "player_count": "Nombre de joueurs", "rounds_per_game": "Manches par partie",
                "branching_factor_average": "Facteur de branchement moyen",
            },
        )
        fig.write_html(os.path.join(FIGURE_DIR, "player_rounds_branching_bubble.html"))
        fig.show()


## Volatilité de la Révolution selon le préset de règles et le nombre de joueurs (heatmap)

In [ ]:
if not manifest_df.empty and {"rule_preset", "player_count", "e_rev_volatility"}.issubset(manifest_df.columns):
    pivot_e_rev = manifest_df.pivot_table(
        index="player_count", columns="rule_preset", values="e_rev_volatility", aggfunc="mean"
    )
    if not pivot_e_rev.empty:
        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(pivot_e_rev, annot=True, fmt=".2f", cmap="crest", ax=ax)
        ax.set_xlabel("Préset de règles")
        ax.set_ylabel("Nombre de joueurs")
        fig.tight_layout()
        fig.savefig(os.path.join(FIGURE_DIR, "e_rev_volatility_preset_heatmap.png"), dpi=150)
        plt.show()


## Distribution de la puissance résultante selon la taille de combinaison (ridgeline)

In [ ]:
if not action_played_df.empty and "resulting_power" in action_played_df.columns:
    played = action_played_df[
        (action_played_df["action_type"] == "ACTION_PLAY") & (action_played_df["resulting_power"].notna())
    ].copy()
    played["combo_size"] = played["cards_played"].apply(len)
    size_buckets = sorted(played["combo_size"].unique())[:8]
    if size_buckets:
        fig, axes = plt.subplots(len(size_buckets), 1, figsize=(8, 1.2 * len(size_buckets)), sharex=True)
        axes = np.atleast_1d(axes)
        for ax, size in zip(axes, size_buckets):
            subset = played.loc[played["combo_size"] == size, "resulting_power"]
            if len(subset) > 1:
                sns.kdeplot(subset, fill=True, ax=ax, alpha=0.6)
            ax.set_ylabel("")
            ax.text(0.01, 0.5, f"taille {size}", transform=ax.transAxes, va="center", fontsize=8)
        axes[-1].set_xlabel("Puissance résultante")
        fig.tight_layout()
        fig.savefig(os.path.join(FIGURE_DIR, "resulting_power_by_combo_size_ridgeline.png"), dpi=150)
        plt.show()
